# MVP Pipeline: Adaptive Augmentations for Small Objects

This notebook is the main entrypoint for testing MVP in Google Colab.

Pipeline steps:
1. Install dependencies.
2. Validate dataset structure.
3. Analyze dataset and save stats.
4. Generate adaptive augmentation policy.
5. (Optional) Run training modes.
6. (Optional) Convert to COCO and run COCOeval.

In [1]:
# Install dependencies in Colab runtime.
%pip install -q ultralytics albumentations opencv-python pycocotools pyyaml numpy pandas matplotlib tqdm gdown


In [2]:
from pathlib import Path
import sys

# 1) Попробуем найти проект в типовых местах
candidates = [
    Path("/content/small_objects_auto_aug"),
    Path("/content/drive/MyDrive/small_objects_auto_aug"),
    Path.cwd(),
    Path.cwd().parent,
]

PROJECT_ROOT = None
for p in candidates:
    if (p / "src").exists() and (p / "configs").exists():
        PROJECT_ROOT = p
        break

# 2) Если не нашли — клонируем в /content (подставь свой URL)
if PROJECT_ROOT is None:
    import subprocess
    repo_url = "https://github.com/s44w/small_objects_auto_aug.git"  # <-- замени
    subprocess.run(["git", "clone", repo_url, "/content/small_objects_auto_aug"], check=True)
    PROJECT_ROOT = Path("/content/small_objects_auto_aug")

sys.path.insert(0, str(PROJECT_ROOT))
print("Using project root:", PROJECT_ROOT)


Using project root: /content/small_objects_auto_aug


In [3]:
# Helpers for robust dataset path checks.
from pathlib import Path

def has_yolo_structure(root: Path) -> bool:
    req = [
        root / 'images' / 'train',
        root / 'images' / 'val',
        root / 'labels' / 'train',
        root / 'labels' / 'val',
    ]
    return all(p.exists() for p in req)


In [4]:
# Imports and configuration.
from src.utils.io import load_yaml
from src.data.visdrone_manager import validate_visdrone_yolo_structure, save_validation_report
from src.analysis.dataset_analyzer import DatasetAnalyzerConfig, analyze_dataset
from src.policy.rule_engine import RuleEngineConfig, generate_policy_from_stats, save_policy_artifacts

project_cfg = load_yaml(PROJECT_ROOT / 'configs' / 'project_config.yaml')

# Keep config root as fallback only; actual dataset_root is resolved in bootstrap cell.
dataset_root_cfg = Path(project_cfg['dataset']['root'])
if not dataset_root_cfg.is_absolute():
    dataset_root_cfg = PROJECT_ROOT / dataset_root_cfg

splits = tuple(project_cfg['dataset'].get('splits', ['train', 'val']))
image_extensions = tuple(project_cfg['dataset'].get('image_extensions', ['.jpg', '.jpeg', '.png', '.bmp']))

stats_dir = PROJECT_ROOT / 'artifacts' / 'stats'
policy_dir = PROJECT_ROOT / 'artifacts' / 'policy'
stats_dir.mkdir(parents=True, exist_ok=True)
policy_dir.mkdir(parents=True, exist_ok=True)

print('Config dataset root (fallback):', dataset_root_cfg)


Config dataset root (fallback): /content/small_objects_auto_aug/datasets/visdrone_yolo


In [5]:
# Dataset bootstrap from Google Drive with auto-discovery + auto-repair from ZIPs.
from pathlib import Path
import shutil
import zipfile
import tempfile

from google.colab import drive
from src.data.visdrone_manager import prepare_visdrone_auto, validate_visdrone_yolo_structure

PROJECT_ROOT = PROJECT_ROOT if 'PROJECT_ROOT' in globals() else Path('/content/small_objects_auto_aug')
DEFAULT_ROOT = PROJECT_ROOT / 'datasets' / 'visdrone_yolo'

drive.mount('/content/drive', force_remount=False)
DRIVE_ROOT = Path('/content/drive/MyDrive')
dataset_root_cfg = globals().get('dataset_root_cfg', None)

def is_yolo_ready(root: Path) -> bool:
    req = [
        root / 'images' / 'train',
        root / 'images' / 'val',
        root / 'labels' / 'train',
        root / 'labels' / 'val',
    ]
    return all(p.exists() for p in req)

def is_raw_ready(root: Path) -> bool:
    # Canonical markers expected by conversion flow.
    if (root / 'VisDrone2019-DET-train').exists() and (root / 'VisDrone2019-DET-val').exists():
        return True

    # Flexible markers for minor naming variations.
    train_like = list(root.glob('*DET*train*')) + list(root.glob('*train*'))
    val_like = list(root.glob('*DET*val*')) + list(root.glob('*val*'))
    if any(p.is_dir() for p in train_like) and any(p.is_dir() for p in val_like):
        return True

    return False

def find_raw_root(drive_root: Path):
    candidates = [
        drive_root / 'datasets' / 'visdrone_raw',
        drive_root / 'visdrone_raw',
        drive_root / 'datasets' / 'VisDrone',
        drive_root / 'VisDrone',
    ]
    for c in candidates:
        if c.exists() and is_raw_ready(c):
            return c

    for t in drive_root.rglob('*DET*train*'):
        if not t.is_dir():
            continue
        parent = t.parent
        if is_raw_ready(parent):
            return parent
    return None

def find_yolo_root(drive_root: Path):
    candidates = [
        drive_root / 'datasets' / 'visdrone_yolo',
        drive_root / 'visdrone_yolo',
        DEFAULT_ROOT,
        dataset_root_cfg if dataset_root_cfg is not None else None,
    ]
    for c in candidates:
        if c is not None and c.exists() and is_yolo_ready(c):
            return c

    for images_train in drive_root.rglob('images/train'):
        root = images_train.parent.parent
        if is_yolo_ready(root):
            return root
    return None

def rebuild_raw_from_zips(zip_root: Path, raw_root: Path) -> bool:
    if not zip_root.exists():
        return False

    zip_files = sorted(zip_root.glob('*.zip'))
    if not zip_files:
        return False

    raw_root.mkdir(parents=True, exist_ok=True)
    print(f'[INFO] Rebuilding raw structure from ZIPs: {zip_root} -> {raw_root}')

    for z in zip_files:
        split_name = z.stem
        target_dir = raw_root / split_name
        if target_dir.exists():
            print(f'[SKIP] already prepared: {target_dir.name}')
            continue

        print(f'[UNZIP] {z.name} -> {target_dir.name}')
        with tempfile.TemporaryDirectory() as td:
            tmp = Path(td)
            with zipfile.ZipFile(z, 'r') as zf:
                zf.extractall(tmp)

            entries = [p for p in tmp.iterdir()]
            if len(entries) == 1 and entries[0].is_dir():
                shutil.move(str(entries[0]), str(target_dir))
            else:
                target_dir.mkdir(parents=True, exist_ok=True)
                for item in entries:
                    shutil.move(str(item), str(target_dir / item.name))

    return True

dataset_root = find_yolo_root(DRIVE_ROOT)
if dataset_root is not None:
    print(f'[OK] Found YOLO dataset: {dataset_root}')
else:
    raw_root = find_raw_root(DRIVE_ROOT)

    if raw_root is None:
        zip_candidates = [
            DRIVE_ROOT / 'datasets' / 'visdrone_zips',
            DRIVE_ROOT / 'visdrone_zips',
        ]
        repaired = False
        for zc in zip_candidates:
            repaired = rebuild_raw_from_zips(zc, DRIVE_ROOT / 'datasets' / 'visdrone_raw')
            if repaired:
                break

        raw_root = find_raw_root(DRIVE_ROOT)

    if raw_root is None:
        preview = []
        ds = DRIVE_ROOT / 'datasets'
        if ds.exists():
            preview = sorted([p.name for p in ds.iterdir()])[:50]
        raise FileNotFoundError(
            'VisDrone raw/YOLO dataset not found in Drive.\n'
            f'Checked under: {DRIVE_ROOT}\n'
            f'datasets/ preview: {preview}\n'
            'Expected raw markers like *DET*train* and *DET*val*'
        )

    target_yolo = DRIVE_ROOT / 'datasets' / 'visdrone_yolo'
    target_yolo.mkdir(parents=True, exist_ok=True)
    print(f'[INFO] Converting raw VisDrone -> YOLO: {raw_root} -> {target_yolo}')
    prepare_visdrone_auto(raw_visdrone_root=raw_root, output_root=target_yolo)
    dataset_root = target_yolo

DEFAULT_ROOT.parent.mkdir(parents=True, exist_ok=True)
if dataset_root != DEFAULT_ROOT:
    if DEFAULT_ROOT.exists() or DEFAULT_ROOT.is_symlink():
        try:
            if DEFAULT_ROOT.is_symlink() or DEFAULT_ROOT.is_file():
                DEFAULT_ROOT.unlink()
            else:
                shutil.rmtree(DEFAULT_ROOT, ignore_errors=True)
        except Exception as e:
            print(f'[WARN] Could not clear default root: {e}')

    try:
        DEFAULT_ROOT.symlink_to(dataset_root, target_is_directory=True)
        print(f'[OK] Linked {DEFAULT_ROOT} -> {dataset_root}')
    except Exception as e:
        print(f'[WARN] Symlink not created: {e}')

report = validate_visdrone_yolo_structure(dataset_root, splits=('train', 'val'))
print('is_valid:', report['is_valid'])
print('train images:', report['splits']['train']['num_images'])
print('val images:', report['splits']['val']['num_images'])
print('Using dataset_root:', dataset_root)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
[INFO] Rebuilding raw structure from ZIPs: /content/drive/MyDrive/datasets/visdrone_zips -> /content/drive/MyDrive/datasets/visdrone_raw
[SKIP] already prepared: VisDrone2019-DET-test-challenge
[UNZIP] VisDrone2019-DET-test-dev.zip -> VisDrone2019-DET-test-dev
[SKIP] already prepared: VisDrone2019-DET-train
[UNZIP] VisDrone2019-DET-val.zip -> VisDrone2019-DET-val
[INFO] Converting raw VisDrone -> YOLO: /content/drive/MyDrive/datasets/visdrone_raw -> /content/drive/MyDrive/datasets/visdrone_yolo
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


RuntimeError: Ultralytics converter is unavailable. Install ultralytics and retry.

In [ ]:
# Quick smoke check (fast): subset -> analyze -> policy -> artifact assertions.
# This is enough to verify project operability without heavy training/eval.
from pathlib import Path
import json

from src.data.subset_builder import build_yolo_subset
from src.analysis.dataset_analyzer import DatasetAnalyzerConfig, analyze_dataset
from src.policy.rule_engine import RuleEngineConfig, generate_policy_from_stats, save_policy_artifacts
from src.data.visdrone_manager import validate_visdrone_yolo_structure

RUN_SMOKE = True

if RUN_SMOKE:
    SMOKE_TRAIN_IMAGES = 60
    SMOKE_VAL_IMAGES = 20
    SMOKE_SPLITS = ('train',)
    SMOKE_SEED = 42

    subset_root = PROJECT_ROOT / 'datasets' / 'visdrone_smoke'
    subset_report = build_yolo_subset(
        dataset_root=dataset_root,
        output_root=subset_root,
        train_images=SMOKE_TRAIN_IMAGES,
        val_images=SMOKE_VAL_IMAGES,
        seed=SMOKE_SEED,
        clean_output=True,
    )
    print('subset:', subset_report)

    val_report = validate_visdrone_yolo_structure(subset_root, splits=('train', 'val'))
    print('subset valid:', val_report['is_valid'])

    smoke_stats_dir = PROJECT_ROOT / 'artifacts' / 'smoke' / 'stats'
    analyzer_cfg = DatasetAnalyzerConfig(
        small_max_area=float(project_cfg['analysis']['small_max_area']),
        medium_max_area=float(project_cfg['analysis']['medium_max_area']),
        tiny_max_area=float(project_cfg['analysis']['tiny_max_area']),
        image_extensions=tuple(project_cfg['dataset'].get('image_extensions', ['.jpg', '.jpeg', '.png', '.bmp'])),
        generate_plots=False,
    )
    smoke_stats = analyze_dataset(
        dataset_root=subset_root,
        output_dir=smoke_stats_dir,
        splits=SMOKE_SPLITS,
        config=analyzer_cfg,
    )

    smoke_policy_dir = PROJECT_ROOT / 'artifacts' / 'smoke' / 'policy'
    rule_cfg = RuleEngineConfig.from_project_config(project_cfg)
    smoke_policy, smoke_decision_report = generate_policy_from_stats(smoke_stats, cfg=rule_cfg)
    smoke_paths = save_policy_artifacts(smoke_policy, smoke_decision_report, output_dir=smoke_policy_dir)

    required_files = [
        smoke_stats_dir / 'dataset_stats.json',
        smoke_stats_dir / 'dataset_stats.csv',
        smoke_policy_dir / 'policy_adaptive.json',
        smoke_policy_dir / 'policy_adaptive.yaml',
        smoke_policy_dir / 'decision_report.json',
    ]
    for p in required_files:
        assert p.exists(), f'Missing artifact: {p}'

    print('\n[OK] Smoke check passed')
    print('dataset_stats:', smoke_stats_dir / 'dataset_stats.json')
    print('policy_json:', smoke_policy_dir / 'policy_adaptive.json')
    print('decision_report:', smoke_policy_dir / 'decision_report.json')
    print('selected albu transforms:', [x['name'] for x in smoke_policy['albumentations_spec']])
    print('ultralytics args:', json.dumps(smoke_policy['ultralytics_args'], ensure_ascii=False, indent=2))
else:
    print('Smoke is skipped. Set RUN_SMOKE=True to enable.')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


FileNotFoundError: Не найден ни готовый YOLO датасет, ни raw VisDrone в Drive. Положи данные в один из candidate paths.

In [ ]:
# Step 1-2 (optional full run): validate and analyze full dataset.
RUN_FULL_ANALYSIS = False

if RUN_FULL_ANALYSIS:
    validation_report = validate_visdrone_yolo_structure(
        dataset_root=dataset_root,
        splits=splits,
        image_extensions=image_extensions,
    )
    save_validation_report(validation_report, stats_dir / 'validation_report.json')
    print('Validation is_valid =', validation_report['is_valid'])

    analyzer_cfg = DatasetAnalyzerConfig(
        small_max_area=float(project_cfg['analysis']['small_max_area']),
        medium_max_area=float(project_cfg['analysis']['medium_max_area']),
        tiny_max_area=float(project_cfg['analysis']['tiny_max_area']),
        image_extensions=image_extensions,
        generate_plots=bool(project_cfg['analysis'].get('generate_plots', True)),
    )
    stats = analyze_dataset(
        dataset_root=dataset_root,
        output_dir=stats_dir,
        splits=splits,
        config=analyzer_cfg,
    )
    print('Stats saved to:', stats_dir)
else:
    print('Full analysis is skipped. Set RUN_FULL_ANALYSIS=True to enable.')


Validation is_valid = True


FileNotFoundError: Missing split images directory: /content/small_objects_auto_aug/datasets/visdrone_yolo/images/train

In [ ]:
# Step 3 (optional full run): generate adaptive policy from full stats.
RUN_FULL_POLICY = False

if RUN_FULL_POLICY:
    rule_cfg = RuleEngineConfig.from_project_config(project_cfg)
    policy, decision_report = generate_policy_from_stats(stats, cfg=rule_cfg)
    paths = save_policy_artifacts(policy=policy, decision_report=decision_report, output_dir=policy_dir)

    print('Policy JSON:', paths['policy_json'])
    print('Policy YAML:', paths['policy_yaml'])
    print('Decision report:', paths['decision_report'])
    print('Fired rules:', len(decision_report.get('fired_rules', [])))
else:
    print('Full policy generation is skipped. Set RUN_FULL_POLICY=True to enable.')


In [ ]:
# Step 4 (optional): run training suite.
# Switch RUN_TRAINING to True when your data.yaml and GPU setup are ready.
RUN_TRAINING = False

if RUN_TRAINING:
    from src.training.train_runner import TrainRunConfig, run_mvp_training_suite

    train_cfg = TrainRunConfig(
        data_yaml=str(project_cfg['training']['data_yaml']),
        model=str(project_cfg['training']['model']),
        epochs=int(project_cfg['training']['epochs_final']),
        imgsz=int(project_cfg['training']['imgsz']),
        batch=int(project_cfg['training']['batch']),
        device=project_cfg['training'].get('device'),
        workers=int(project_cfg['training']['workers']),
        fraction=1.0,
        project_dir=str(PROJECT_ROOT / 'runs'),
        seed=int(project_cfg['project']['seed']),
        deterministic=bool(project_cfg['project']['deterministic']),
        rect=bool(project_cfg['training'].get('rect', False)),
        multi_scale=bool(project_cfg['training'].get('multi_scale', False)),
        baseline_disable_albumentations=bool(project_cfg['training'].get('baseline_disable_albumentations', True)),
    )

    run_dirs = run_mvp_training_suite(
        config=train_cfg,
        baseline_yaml_path=PROJECT_ROOT / 'configs' / 'baseline.yaml',
        manual_yaml_path=PROJECT_ROOT / 'configs' / 'manual.yaml',
        adaptive_policy_json_path=policy_dir / 'policy_adaptive.json',
        run_ablation=True,
    )
    print(run_dirs)
else:
    print('Training is skipped. Set RUN_TRAINING=True to enable.')

In [ ]:
# Step 5 (optional): COCO conversion and COCOeval.
# Provide directory with YOLO prediction labels (*.txt) for val split.
RUN_EVAL = False
PRED_LABELS_DIR = PROJECT_ROOT / 'runs' / 'detect' / 'predict' / 'labels'  # edit if needed

if RUN_EVAL:
    from src.evaluation.coco_converter import convert_yolo_gt_to_coco, convert_yolo_pred_txt_to_coco
    from src.evaluation.coco_eval_runner import run_coco_eval
    from src.evaluation.metrics_report import build_markdown_report

    eval_dir = PROJECT_ROOT / 'artifacts' / 'eval'
    eval_dir.mkdir(parents=True, exist_ok=True)

    coco_gt_path = eval_dir / 'coco_gt.json'
    coco_dt_path = eval_dir / 'coco_dt.json'

    convert_yolo_gt_to_coco(
        images_dir=dataset_root / 'images' / 'val',
        labels_dir=dataset_root / 'labels' / 'val',
        class_names=project_cfg['dataset']['visdrone_classes'],
        output_path=coco_gt_path,
        image_extensions=image_extensions,
    )
    convert_yolo_pred_txt_to_coco(
        pred_labels_dir=PRED_LABELS_DIR,
        images_dir=dataset_root / 'images' / 'val',
        output_path=coco_dt_path,
        image_extensions=image_extensions,
    )

    metrics = run_coco_eval(
        coco_gt_path=coco_gt_path,
        coco_dt_path=coco_dt_path,
        output_path=eval_dir / 'coco_eval.json',
        use_tiny_eval=True,
        tiny_max_area=float(project_cfg['analysis']['tiny_max_area']),
    )

    report_path = build_markdown_report({'adaptive': metrics}, PROJECT_ROOT / 'artifacts' / 'reports' / 'mvp_report.md')
    print(metrics)
    print('Report:', report_path)
else:
    print('Evaluation is skipped. Set RUN_EVAL=True to enable.')